# Species Abundance Modeling: Wind Energy Effects

**Authors:** Samantha A. McGarrigle<sup>1\*</sup>, Ellery Vaughn Lassiter<sup>1</sup>, Karen Maguire<sup>2</sup>, Rich Iovanna<sup>3</sup>, Jay Diffendorfer<sup>4</sup>, Anthony Lopez<sup>5</sup>, Kyle Reed<sup>2</sup>, Courtney Duchardt<sup>1,6</sup>, Scott Loss<sup>1</sup>  
**Journal:** [Journal Name]  
**Article page:** [Add link here when available]  
**DOI:** [Add DOI here when available]  

**Affiliations:**  
<sup>1</sup> Department of Natural Resource Ecology and Management, Oklahoma State University, Stillwater, Oklahoma, USA  
<sup>2</sup> USDA Economic Research Service, U.S. Department of Agriculture, Kansas City, Missouri, USA  
<sup>3</sup> USDA Farm Production and Conservation Mission Area, Economic and Policy Analysis Division, Washington D.C., USA  
<sup>4</sup> United States Geological Survey, Geosciences and Environmental Change Science Center, Lakewood, CO, USA  
<sup>5</sup> U.S. Department of Energy, National Renewable Energy Laboratory, Golden, Colorado, USA  
<sup>6</sup> School of Natural Resources and the Environment, University of Arizona, Tucson, Arizona, USA  

<sup>\*</sup> Corresponding author

---

This notebook contains the full analytical pipeline for the wind energy only analysis. It is structured to mirror the Methods section of the paper and walks through:

1. eBird data extraction and preparation
2. Spatial filtering and subsampling
3. Joining land cover and wind turbine datasets
4. Wind variable categorization
5. Correlation matrix
6. Standardization and train/test splitting
7. Distribution family evaluation (null models)
8. Univariate variable screening (effort, land cover, wind)
9. Full wind candidate model fitting (with effort and land cover)
10. Model diagnostics
11. Figure generation

> **Note:** All species names, file paths, and spatial grain parameters have been generalized. Replace placeholder values (e.g., `species_name_here`, `wind_variable`) with your specific inputs before running.

---
## Part 1: eBird Data Extraction and Preparation

Raw eBird Basic Dataset (EBD) files are filtered using the `auk` package. Observations are zero-filled to include non-detection checklists, then cleaned and filtered by effort constraints.

In [ ]:
# Libraries
# Data manipulation & tidy tools
library(tidyverse)
library(lubridate)
# Spatial & raster data
library(sf)
library(raster)
library(dggridR)
library(rnaturalearth)
# Plotting & graphics
library(ggpubr)
library(gridExtra)
library(viridis)
library(fields)
library(corrplot)
# Statistical modeling
library(mgcv)
library(pscl)
library(MASS)
library(FSA)
library(fitdistrplus)
library(hglm)
library(zigam)
library(grplasso)
# Model comparison & diagnostics
library(AICcmodavg)
library(MuMIn)
library(DescTools)
library(pdp)
# eBird specific
library(auk)
library(ebirdst)

# Resolve namespace conflicts
select <- dplyr::select

# Set working directory
setwd("path/to/your/project/data")

In [ ]:
# Setup auk eBird data (update paths as appropriate)
ebd <- auk_ebd("raw_data/ebd_species_raw.txt", file_sampling = "raw_data/ebd_sampling_raw.txt")

# Initial filters (replace species name with your target species)
ebd_filters <- ebd %>%
  auk_species("species_name_here") %>%
  auk_year(year = 2012:2021) %>%
  auk_date(date = c("*-06-14", "*-08-03")) %>%
  auk_protocol(protocol = c("Stationary", "Traveling")) %>%
  auk_complete()

# Output directory and paths
data_dir <- "data"
if (!dir.exists(data_dir)) dir.create(data_dir)

f_ebd      <- file.path(data_dir, "ebd_species_breed.txt")
f_sampling <- file.path(data_dir, "ebd_sampling_species_breed.txt")

# Filter and write if not already done
if (!file.exists(f_ebd)) {
  auk_filter(ebd_filters, file = f_ebd, file_sampling = f_sampling)
}

In [ ]:
# Zero-fill to include non-detections
ebd_zf <- auk_zerofill(f_ebd, f_sampling, collapse = TRUE)

# Helper: convert HH:MM:SS to decimal hours
time_to_decimal <- function(x) {
  x <- hms(x, quiet = TRUE)
  hour(x) + minute(x) / 60 + second(x) / 3600
}

# Clean and transform variables
ebd_zf <- ebd_zf %>%
  mutate(
    observation_count = if_else(observation_count == "X", NA_character_, observation_count),
    observation_count = as.integer(observation_count),
    effort_distance_km = if_else(protocol_type != "Traveling", 0, effort_distance_km),
    time_observations_started = time_to_decimal(time_observations_started),
    year = year(observation_date),
    day_of_year = yday(observation_date)
  )

# Filter by effort constraints
ebd_filtered <- ebd_zf %>%
  filter(
    duration_minutes   <= 300,
    effort_distance_km <= 5,
    number_observers   <= 10
  )

# Select relevant columns
ebird_data <- ebd_filtered %>%
  select(
    checklist_id, observer_id, sampling_event_identifier,
    scientific_name, observation_count, species_observed,
    state_code, locality_id, latitude, longitude,
    protocol_type, all_species_reported,
    observation_date, year, day_of_year,
    time_observations_started,
    duration_minutes, effort_distance_km, number_observers
  )

---
## Part 2: Spatial Filtering, Hexagonal Grid Assignment, and Subsampling

Checklists are clipped to the species breeding range within the US border. A discrete global grid (dggridR) assigns each checklist to a spatial cell. Data are then summarized per cell-year and subsampled to equalize annual effort.

In [ ]:
# Load spatial boundaries
USborder           <- st_read("shapefiles/US_border.shp") %>% st_transform(crs = 4326)
rangemap_allseasons <- st_read("range_maps/species_range.gpkg")

# Subset to breeding season and clip to US
rangemap_season <- filter(rangemap_allseasons, season == "breeding")
rangemap        <- st_intersection(rangemap_season, USborder)

# Convert checklists to sf points and intersect with range
ebird_sf       <- st_as_sf(ebird_data, coords = c("longitude", "latitude"), crs = 4326)
ebird_filtered <- st_intersection(ebird_sf, rangemap) %>% as_tibble()

# Extract lat/lon from geometry
ebird_filtered <- ebird_filtered %>%
  mutate(
    longitude = map_dbl(geometry, 1),
    latitude  = map_dbl(geometry, 2)
  )

# Remove checklists with missing counts
ebird_filtered <- ebird_filtered %>% filter(!is.na(observation_count))

In [ ]:
# Construct hexagonal grid (~4.5 km spacing) and assign cells
dggs <- dgconstruct(spacing = 9)

ebird_filtered <- ebird_filtered %>%
  mutate(
    cell = dgGEO_to_SEQNUM(dggs, longitude, latitude)$seqnum,
    week = week(observation_date)
  )

# Summarize: mean abundance and effort per cell-year
ebird_summary <- ebird_filtered %>%
  mutate(cell_year = paste(cell, year, sep = "_")) %>%
  group_by(cell_year) %>%
  summarise(
    meanCount     = mean(observation_count),
    year          = max(year),
    meanStartTime = mean(time_observations_started),
    meanSampDur   = mean(duration_minutes),
    meanDIST      = mean(effort_distance_km),
    meanNumObs    = mean(number_observers)
  ) %>%
  ungroup() %>%
  mutate(species_observed = if_else(meanCount > 0, "TRUE", "FALSE"))

# Subsample to equalize effort across years
min_year_count <- min(table(ebird_summary$year))
ebird_summary_ss <- ebird_summary %>%
  group_by(year) %>%
  sample_n(size = min_year_count) %>%
  ungroup()

# Save cleaned data
write_csv(ebird_summary_ss, file.path(data_dir, "species_breed_finaldata.csv"))

---
## Part 3: Joining Ancillary Datasets

The core eBird summary is joined with land cover and wind turbine datasets. Prevalence is inspected after joining.

In [ ]:
# Load bird summary and keep relevant columns
spec_raw_spatialgrain   <- read_csv(file.choose())  # species_season_finaldata.csv
spec_clean_spatialgrain <- spec_raw_spatialgrain[, c(1:11, 36:45)]

# Join land cover (proportion of landscape by class)
landcover <- read.csv("path/to/landcover4.5_cell_PLAND.csv", header = TRUE)
landcover <- landcover %>% select(-seqnum, -year)
spec_joined_spatialgrain <- left_join(spec_clean_spatialgrain, landcover, by = "cell_year")

# Join wind turbine data (fill NAs with 0 = no turbines)
turbines <- read.csv(file.choose(), header = TRUE)  # WRD_foranalysis.csv
spec_joined_spatialgrain <- left_join(spec_joined_spatialgrain, turbines, by = "cell_year") %>%
  mutate(WindCount = ifelse(is.na(WindCount), 0, WindCount))

# Inspect prevalence
spec_joined_spatialgrain_ss %>%
  count(species_observed) %>%
  mutate(percent = n / sum(n))

# Final dataset for downstream analyses
spec_final <- spec_joined_spatialgrain_ss

---
## Part 4: Wind Variable Categorization

Continuous wind turbine attributes (count, age, hub height, rotor swept area, capacity) are binned into ordered categorical variables to allow flexible nonlinear or step-wise effects in GAMs.

In [ ]:
# Presence/absence
spec_joined_spatialgrain$WindPA <- as.factor(
  ifelse(spec_joined_spatialgrain$WindCount > 0, "Y", "N")
)

# Age bins
spec_joined_spatialgrain$WindAge_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindAge <= 2, "0-2",
  ifelse(spec_joined_spatialgrain$WindAge <= 4, "3-4",
  ifelse(spec_joined_spatialgrain$WindAge <= 6, "5-6",
  ifelse(spec_joined_spatialgrain$WindAge <= 8, "7-8", "9+")))))
spec_joined_spatialgrain$WindAge_cat <- factor(
  spec_joined_spatialgrain$WindAge_cat, levels = c("None","0-2","3-4","5-6","7-8","9+")
)

# Hub height bins
spec_joined_spatialgrain$WindHeight_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindHeight <= 20,  "1-20",
  ifelse(spec_joined_spatialgrain$WindHeight <= 40,  "21-40",
  ifelse(spec_joined_spatialgrain$WindHeight <= 60,  "41-60",
  ifelse(spec_joined_spatialgrain$WindHeight <= 80,  "61-80", "81-100+")))))
spec_joined_spatialgrain$WindHeight_cat <- factor(
  spec_joined_spatialgrain$WindHeight_cat,
  levels = c("None","1-20","21-40","41-60","61-80","81-100+")
)

# Rotor swept area bins (m²)
spec_joined_spatialgrain$WindRSA_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindRSA <= 5000,  "0-5000",
  ifelse(spec_joined_spatialgrain$WindRSA <= 10000, "5001-10000",
  ifelse(spec_joined_spatialgrain$WindRSA <= 15000, "10001-15000", "15000+"))))
spec_joined_spatialgrain$WindRSA_cat <- factor(
  spec_joined_spatialgrain$WindRSA_cat,
  levels = c("None","0-5000","5001-10000","10001-15000","15000+")
)

# Capacity bins (kW)
spec_joined_spatialgrain$WindCap_cat <- ifelse(spec_joined_spatialgrain$WindPA == "N", "None",
  ifelse(spec_joined_spatialgrain$WindCap <= 1000, "0-1000",
  ifelse(spec_joined_spatialgrain$WindCap <= 2000, "1001-2000",
  ifelse(spec_joined_spatialgrain$WindCap <= 3000, "2001-3000",
  ifelse(spec_joined_spatialgrain$WindCap <= 4000, "3001-4000", "4001+")))))
spec_joined_spatialgrain$WindCap_cat <- factor(
  spec_joined_spatialgrain$WindCap_cat,
  levels = c("None","0-1000","1001-2000","2001-3000","3001-4000","4001+")
)

# Drop rows missing key wind attributes
spec_filtered_spatialgrain <- spec_joined_spatialgrain %>%
  drop_na(WindHeight_cat, WindCap_cat, WindRSA_cat)

---
## Part 5: Correlation Matrix

A Pearson correlation matrix is computed across all candidate covariates to identify multicollinearity prior to model construction. Results are saved for inspection.

In [ ]:
cortest <- spec_filtered_spatialgrain %>%
  select(
    WindCount,
    Developed, Cropland, GrassShrub, TreeCover, Water, Wetland, Barren, IceSnow,
    meanStartTime, meanSampDur, meanDIST, meanNumObs
  )

corr_matrix <- cor(cortest, use = "complete.obs")

# Reorder columns for visual grouping
corr_matrix <- corr_matrix[, c(
  "Developed","Cropland","GrassShrub","TreeCover","Water","Wetland","Barren","IceSnow",
  "meanStartTime","meanSampDur","meanDIST","meanNumObs","WindCount"
)]

write.csv(corr_matrix, "spec_corr_matrix.csv", row.names = TRUE)

# Visualize
corrplot(corr_matrix, method = "color", tl.cex = 0.6)

---
## Part 6: Standardization and Train/Test Split

All continuous covariates are z-score standardized. Scaling parameters (mean, SD) are saved to allow back-transformation for figure generation. Data are split 80/20 into training and test sets.

In [ ]:
scaled_cols <- spec_filtered_spatialgrain %>%
  select(
    meanStartTime, meanSampDur, meanDIST, meanNumObs,
    Developed, Cropland, GrassShrub, TreeCover, Water, Wetland, Barren, IceSnow,
    WindCount, Lat, Long
  )

# Save scaling parameters for back-transformation
scaling_params <- scaled_cols %>%
  summarise(across(where(is.numeric),
    list(mean = ~ mean(., na.rm = TRUE), sd = ~ sd(., na.rm = TRUE)))) %>%
  pivot_longer(everything(), names_to = c("variable", "statistic"), names_sep = "_",
               values_to = "value") %>%
  pivot_wider(names_from = "statistic", values_from = "value")

scaled_data <- scaled_cols %>% mutate(across(where(is.numeric), scale))

spec_scaled_spatialgrain <- cbind(
  meanCount       = spec_filtered_spatialgrain$meanCount,
  meanCount_round = spec_filtered_spatialgrain$meanCount_round,
  WindAge         = spec_filtered_spatialgrain$WindAge_cat,
  WindHeight      = spec_filtered_spatialgrain$WindHeight_cat,
  WindCap         = spec_filtered_spatialgrain$WindCap_cat,
  WindRSA         = spec_filtered_spatialgrain$WindRSA_cat,
  WindPA          = spec_filtered_spatialgrain$WindPA,
  scaled_data
)

In [ ]:
# Select model inputs and perform 80/20 random split
spec_split_spatialgrain <- spec_scaled_spatialgrain %>%
  select(
    meanCount, meanCount_round,
    meanStartTime, meanSampDur, meanDIST, meanNumObs,
    Developed, Cropland, GrassShrub, TreeCover, Water, Wetland, Barren, IceSnow,
    WindCount, WindHeight, WindCap, WindAge, WindPA, WindRSA,
    Lat, Long
  ) %>%
  split(if_else(runif(nrow(.)) <= 0.8, "train", "test"))

map_int(spec_split_spatialgrain, nrow)

---
## Part 7: Distribution Family Evaluation Using Null Models

Null GAMs (intercept only) are fit under Poisson, negative binomial, zero-inflated Poisson, and zero-inflated negative binomial families. AIC/BIC are used to select the most appropriate count distribution for the response variable.

In [ ]:
# GAM smoothing parameters
k        <- 5   # knots for most smooths
k_time   <- 7   # knots for cyclic time-of-day smooth
time_knots <- list(meanStartTime = seq(0, 24, length.out = k_time))

Null <- meanCount_round ~ 1

# Negative binomial
start.time <- Sys.time()
gam_nb   <- gam(Null, data = spec_split_spatialgrain$train, family = "nb", knots = time_knots)
cat("NB:", round(Sys.time() - start.time, 2), "sec\n")

# Poisson
start.time <- Sys.time()
gam_pois <- gam(Null, data = spec_split_spatialgrain$train, family = "poisson", knots = time_knots)
cat("Poisson:", round(Sys.time() - start.time, 2), "sec\n")

# Zero-inflated NB
start.time <- Sys.time()
gam_zinb <- zinbgam(Null, pi.formula = ~1, data = spec_split_spatialgrain$train, knots = time_knots)
cat("ZINB:", round(Sys.time() - start.time, 2), "sec\n")

# Zero-inflated Poisson
start.time <- Sys.time()
gam_zip  <- gam(Null, data = spec_split_spatialgrain$train, family = "ziP", knots = time_knots)
cat("ZIP:", round(Sys.time() - start.time, 2), "sec\n")

# Quasi-Poisson (for overdispersion check; no AIC)
gam_qp <- gam(Null, data = spec_split_spatialgrain$train, family = "quasipoisson", knots = time_knots)

# Compare
AIC(gam_nb, gam_pois, gam_zip)
BIC(gam_nb, gam_pois, gam_zip)

# Manual BIC for ZINB (not directly available)
AIC_zinb <- gam_zinb$aic
BIC_zinb <- AIC_zinb + log(nrow(spec_split_spatialgrain$train)) *
            (length(coef(gam_zinb)) - length(fitted(gam_zinb)) + 1)
cat("ZINB AIC:", AIC_zinb, "\nZINB BIC:", BIC_zinb, "\n")

# Quasi-Poisson summary
sum.gam <- summary(gam_qp)
sum.gam$p.pv
sum.gam$s.pv
sum(residuals(gam_qp, type = "response"))

---
## Part 8: Univariate Variable Screening (Effort, Land Cover, Wind)

Each candidate covariate is added individually to a base spatial model to evaluate its marginal contribution via effective degrees of freedom (edf) and AICc.

In [ ]:
# --- Effort variable screening ---
base             <- meanCount ~ s(Lat, k=5) + s(Long, k=5)
effort_SampDur   <- meanCount ~ s(meanSampDur, k=5)   + s(Lat, k=5) + s(Long, k=5)
effort_NumObs    <- meanCount ~ s(meanNumObs, k=5)    + s(Lat, k=5) + s(Long, k=5)
effort_Dist      <- meanCount ~ s(meanDIST, k=5)      + s(Lat, k=5) + s(Long, k=5)
effort_StartTime <- meanCount ~ s(meanStartTime, bs="cc", k=7) + s(Lat, k=5) + s(Long, k=5)
effort_Global    <- meanCount ~ s(meanSampDur, k=5) + s(meanNumObs, k=5) + s(meanDIST, k=5) +
                                s(meanStartTime, bs="cc", k=7) + s(Lat, k=5) + s(Long, k=5)

start.time <- Sys.time()
m_effort_base      <- gam(base,             data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_effort_base:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_effort_base)

start.time <- Sys.time()
m_effort_SampDur   <- gam(effort_SampDur,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_effort_SampDur:",   round(Sys.time()-start.time,2), "sec\n"); summary(m_effort_SampDur)

start.time <- Sys.time()
m_effort_NumObs    <- gam(effort_NumObs,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_effort_NumObs:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_effort_NumObs)

start.time <- Sys.time()
m_effort_Dist      <- gam(effort_Dist,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_effort_Dist:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_effort_Dist)

start.time <- Sys.time()
m_effort_StartTime <- gam(effort_StartTime, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_effort_StartTime:", round(Sys.time()-start.time,2), "sec\n"); summary(m_effort_StartTime)

start.time <- Sys.time()
m_effort_global    <- gam(effort_Global,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_effort_global:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_effort_global)

MuMIn::AICc(m_effort_base, m_effort_SampDur, m_effort_NumObs,
            m_effort_Dist, m_effort_StartTime, m_effort_global)

In [ ]:
# --- Land cover variable screening ---
lc_base      <- meanCount ~ s(Lat,k=5) + s(Long,k=5)
lc_developed <- meanCount ~ s(Developed,k=5) + s(Lat,k=5) + s(Long,k=5)
lc_cropland  <- meanCount ~ s(Cropland,k=5)  + s(Lat,k=5) + s(Long,k=5)
lc_grasshrub <- meanCount ~ s(GrassShrub,k=5) + s(Lat,k=5) + s(Long,k=5)
lc_treecover <- meanCount ~ s(TreeCover,k=5) + s(Lat,k=5) + s(Long,k=5)
lc_water     <- meanCount ~ s(Water,k=5)     + s(Lat,k=5) + s(Long,k=5)
lc_wetland   <- meanCount ~ s(Wetland,k=5)   + s(Lat,k=5) + s(Long,k=5)
lc_barren    <- meanCount ~ s(Barren,k=5)    + s(Lat,k=5) + s(Long,k=5)
lc_icesnow   <- meanCount ~ s(IceSnow,k=5)   + s(Lat,k=5) + s(Long,k=5)

start.time <- Sys.time()
m_lc_base      <- gam(lc_base,      data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_base:",      round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_base)

start.time <- Sys.time()
m_lc_developed <- gam(lc_developed, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_developed:", round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_developed)

start.time <- Sys.time()
m_lc_cropland  <- gam(lc_cropland,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_cropland:",  round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_cropland)

start.time <- Sys.time()
m_lc_grasshrub <- gam(lc_grasshrub, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_grasshrub:", round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_grasshrub)

start.time <- Sys.time()
m_lc_treecover <- gam(lc_treecover, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_treecover:", round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_treecover)

start.time <- Sys.time()
m_lc_water     <- gam(lc_water,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_water:",     round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_water)

start.time <- Sys.time()
m_lc_wetland   <- gam(lc_wetland,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_wetland:",   round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_wetland)

start.time <- Sys.time()
m_lc_barren    <- gam(lc_barren,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_barren:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_barren)

start.time <- Sys.time()
m_lc_icesnow   <- gam(lc_icesnow,   data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_lc_icesnow:",   round(Sys.time()-start.time,2), "sec\n"); summary(m_lc_icesnow)

MuMIn::AICc(m_lc_base, m_lc_developed, m_lc_cropland, m_lc_grasshrub,
            m_lc_treecover, m_lc_water, m_lc_wetland, m_lc_barren, m_lc_icesnow)

In [ ]:
# --- Wind variable screening ---
Wind_windcount_simple  <- meanCount ~ s(WindCount,k=5)    + s(Lat,k=5) + s(Long,k=5)
Wind_windPA_simple     <- meanCount ~ factor(WindPA)       + s(Lat,k=5) + s(Long,k=5)
Wind_windrsa_simple    <- meanCount ~ factor(WindRSA)      + s(Lat,k=5) + s(Long,k=5)
Wind_windheight_simple <- meanCount ~ factor(WindHeight)   + s(Lat,k=5) + s(Long,k=5)
Wind_windage_simple    <- meanCount ~ factor(WindAge)      + s(Lat,k=5) + s(Long,k=5)
Wind_windcap_simple    <- meanCount ~ factor(WindCap)      + s(Lat,k=5) + s(Long,k=5)

start.time <- Sys.time()
m_wind_count_simple  <- gam(Wind_windcount_simple,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_count_simple:",  round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_count_simple)

start.time <- Sys.time()
m_wind_cap_simple    <- gam(Wind_windcap_simple,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_cap_simple:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_cap_simple)

start.time <- Sys.time()
m_Wind_windPA_simple <- gam(Wind_windPA_simple,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_Wind_windPA_simple:", round(Sys.time()-start.time,2), "sec\n"); summary(m_Wind_windPA_simple)

start.time <- Sys.time()
m_wind_rsa_simple    <- gam(Wind_windrsa_simple,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_rsa_simple:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_rsa_simple)

start.time <- Sys.time()
m_wind_height_simple <- gam(Wind_windheight_simple, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_height_simple:", round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_height_simple)

start.time <- Sys.time()
m_wind_age_simple    <- gam(Wind_windage_simple,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_age_simple:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_age_simple)

MuMIn::AICc(m_wind_count_simple, m_wind_cap_simple, m_Wind_windPA_simple,
            m_wind_rsa_simple, m_wind_height_simple, m_wind_age_simple, m_lc_base)

---
## Part 9: Full Wind Models (with Effort and Land Cover)

Full candidate wind models are fit including all retained effort and land cover covariates. Replace `wind_variable` with the best-supported wind metric identified in Part 8.

In [ ]:
# Replace 'wind_variable' with the selected wind covariate name (e.g., WindCount, WindCap)

base_full <- meanCount ~
  s(meanSampDur,k=5) + s(meanNumObs,k=5) + s(meanDIST,k=5) +
  s(meanStartTime,bs="cc",k=7) +
  s(Developed,k=5) + s(TreeCover,k=5) + s(Water,k=5) + Wetland +
  s(Barren,k=5) + s(IceSnow,k=5) +
  s(Lat,k=5) + s(Long,k=5)

Wind_windcount <- update(base_full, . ~ . + s(WindCount, k=5))
Wind_windcap   <- update(base_full, . ~ . + factor(WindCap))
Wind_windPA    <- update(base_full, . ~ . + factor(WindPA))
Wind_windrsa   <- update(base_full, . ~ . + factor(WindRSA))
Wind_windheight <- update(base_full, . ~ . + factor(WindHeight))
Wind_windage   <- update(base_full, . ~ . + factor(WindAge))

start.time <- Sys.time()
m_wind_count  <- gam(Wind_windcount,  data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_count:",  round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_count)

start.time <- Sys.time()
m_wind_cap    <- gam(Wind_windcap,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_cap:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_cap)

start.time <- Sys.time()
m_Wind_windPA <- gam(Wind_windPA,     data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_Wind_windPA:", round(Sys.time()-start.time,2), "sec\n"); summary(m_Wind_windPA)

start.time <- Sys.time()
m_wind_rsa    <- gam(Wind_windrsa,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_rsa:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_rsa)

start.time <- Sys.time()
m_wind_height <- gam(Wind_windheight, data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_height:", round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_height)

start.time <- Sys.time()
m_wind_age    <- gam(Wind_windage,    data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_age:",    round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_age)

start.time <- Sys.time()
m_wind_base   <- gam(base_full,       data=spec_split_spatialgrain$train, family="nb", knots=time_knots)
cat("m_wind_base:",   round(Sys.time()-start.time,2), "sec\n"); summary(m_wind_base)

MuMIn::AICc(m_wind_count, m_wind_cap, m_Wind_windPA,
            m_wind_rsa, m_wind_height, m_wind_age, m_wind_base)

---
## Part 10: Top Model Diagnostics

Variance components and ANOVA table are inspected for the top wind model. Default GAM diagnostic plots are also produced.

In [ ]:
# Replace m_wind with the top model object identified by AICc above

# Variance components (rescaled)
gam.vcomp(m_wind, rescale = TRUE)

# ANOVA table
anova(m_wind)

# Default diagnostic plots
par(mfrow = c(2, 2))
plot(m_wind)

---
## Part 11: Figure Generation

Predicted abundance from the top wind models is visualized against the focal predictors. Continuous variables are back-transformed using saved scaling parameters for interpretable axes.

In [ ]:
# Back-transform standardized wind variable for plot axis
scaling_params_sub <- scaling_params %>%
  filter(variable == "wind_variable")

spec_split_spatialgrain$test$wind_variable_unscaled <-
  spec_split_spatialgrain$test$wind_variable *
  scaling_params_sub$sd[scaling_params_sub$variable == "wind_variable"] +
  scaling_params_sub$mean[scaling_params_sub$variable == "wind_variable"]

In [ ]:
# Figure 1: Continuous wind variable
pred_wind_nl <- predict.gam(m_wind_nonlinear, newdata=spec_split_spatialgrain$test,
                            type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_wind_nl <- spec_split_spatialgrain$test
pred_df_wind_nl$predicted <- pred_wind_nl$fit

ggplot(pred_df_wind_nl, aes(x=wind_variable_unscaled, y=predicted)) +
  geom_smooth() +
  labs(x="Wind Variable", y="Predicted Abundance") +
  theme_minimal()

In [ ]:
# Figure 2: Categorical wind variable
pred_wind_cat <- predict.gam(m_wind_categorical, newdata=spec_split_spatialgrain$test,
                             type="response", se.fit=TRUE, allow.new.levels=TRUE)
pred_df_wind_cat <- spec_split_spatialgrain$test
pred_df_wind_cat$predicted <- pred_wind_cat$fit

ggplot(pred_df_wind_cat, aes(x=wind_variable, y=predicted)) +
  geom_boxplot() +
  labs(x="Wind Variable", y="Predicted Abundance") +
  theme_minimal()